# Day 20 - Financial Risk Scoring & Classification

## Objectives

- Identify financial risk signals
- Evaluate leverage and debt risk
- Measure profitability risk
- Analyze interest coverage risk
- Integrate financial momentum into risk analysis
- Build a composite financial risk score
- Classify companies into risk categories
- Generate company risk reports

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

REPORT_DIR = "../reports"

print("Day 20 - Financial Risk Scoring")

Day 20 - Financial Risk Scoring


In [2]:
trend_path = "../reports/day19_financial_trends.csv"

trends = pd.read_csv(trend_path)

print("Shape:", trends.shape)

print("\nColumns:")
print(trends.columns.tolist())

trends.head()

Shape: (1065, 19)

Columns:
['company_id', 'year', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'dividend_payout_ratio', 'financial_year', 'return_on_equity_pct_yoy_change', 'debt_to_equity_yoy_change', 'interest_coverage_yoy_change', 'asset_turnover_yoy_change', 'dividend_payout_ratio_yoy_change', 'roe_momentum', 'debt_momentum', 'interest_momentum', 'asset_turnover_momentum', 'financial_momentum_score', 'trend_classification']


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio,financial_year,return_on_equity_pct_yoy_change,debt_to_equity_yoy_change,interest_coverage_yoy_change,asset_turnover_yoy_change,dividend_payout_ratio_yoy_change,roe_momentum,debt_momentum,interest_momentum,asset_turnover_momentum,financial_momentum_score,trend_classification
0,ABB,Dec 2012,22.411128,0.0,NaN,1.822492,17.241379,2012,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,Declining
1,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263,2014,2.715775,0.0,NaN,0.175752,-4.615117,1,0,0,1,2,Improving
2,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755,2015,-0.687202,0.0,NaN,-0.332305,0.037493,0,0,0,-1,-1,Declining
3,ABB,Mar 2016,21.338912,0.0,121.666667,1.617574,11.372549,2016,-3.100789,0.0,NaN,-0.048365,-1.291206,-1,0,0,0,-1,Declining
4,ABB,Mar 2017,19.971161,0.0,199.000000,1.405131,11.191336,2017,-1.367751,0.0,77.333333,-0.212444,-0.181213,0,0,1,-1,0,Declining


In [3]:
latest_data = (
    trends
    .sort_values(
        ["company_id", "financial_year"]
    )
    .groupby("company_id")
    .tail(1)
    .copy()
)

latest_data = latest_data.reset_index(drop=True)

print("Companies:", latest_data["company_id"].nunique())
print("Shape:", latest_data.shape)

latest_data.head()

Companies: 92
Shape: (92, 19)


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio,financial_year,return_on_equity_pct_yoy_change,debt_to_equity_yoy_change,interest_coverage_yoy_change,asset_turnover_yoy_change,dividend_payout_ratio_yoy_change,roe_momentum,debt_momentum,interest_momentum,asset_turnover_momentum,financial_momentum_score,trend_classification
0,ABB,Mar 2024,32.468235,0.022438,121.083333,1.126324,6.078268,2024,2.700355,-0.013007,45.708333,-0.047732,-1.614040,1,0,1,0,2,Improving
1,ADANIENSOL,Mar 2024,9.461277,2.932521,2.063968,0.283696,0.000000,2024,-1.441779,0.015677,0.439373,0.037219,0.000000,0,0,0,0,0,Declining
2,ADANIENT,Mar 2024,8.534650,1.671358,2.497695,0.600432,0.149925,2024,1.206582,0.061725,0.275977,-0.302327,-0.097804,0,0,0,-1,-1,Declining
3,ADANIGREEN,Mar 2024,12.812691,6.595282,1.461846,0.104670,0.000000,2024,-0.508777,-0.828459,-0.245471,-0.011547,0.000000,0,1,0,0,1,Improving
4,ADANIPORTS,Mar 2024,15.354883,0.937322,5.703988,0.228301,0.197433,2024,3.477556,-0.239923,1.071318,0.043054,-0.173555,1,1,1,0,3,Strong Improvement


In [4]:
risk_metrics = [
    "return_on_equity_pct",
    "debt_to_equity",
    "interest_coverage",
    "asset_turnover",
    "financial_momentum_score"
]

available_risk_metrics = [
    col
    for col in risk_metrics
    if col in latest_data.columns
]

print("Available Risk Metrics:")
print(available_risk_metrics)

print("\nMissing Risk Metrics:")
print([
    col
    for col in risk_metrics
    if col not in latest_data.columns
])

Available Risk Metrics:
['return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'financial_momentum_score']

Missing Risk Metrics:
[]


In [5]:
for col in available_risk_metrics:
    latest_data[col] = pd.to_numeric(
        latest_data[col],
        errors="coerce"
    )

print(latest_data[available_risk_metrics].dtypes)

return_on_equity_pct        float64
debt_to_equity              float64
interest_coverage           float64
asset_turnover              float64
financial_momentum_score      int64
dtype: object


In [6]:
if "debt_to_equity" in latest_data.columns:
    latest_data["debt_risk_score"] = np.select(
        [
            latest_data["debt_to_equity"].isna(),
            latest_data["debt_to_equity"] <= 0.5,
            latest_data["debt_to_equity"] <= 1.0,
            latest_data["debt_to_equity"] <= 2.0,
            latest_data["debt_to_equity"] > 2.0
        ],
        [
            2,
            0,
            1,
            2,
            3
        ],
        default=2
    )

latest_data[
    [
        "company_id",
        "debt_to_equity",
        "debt_risk_score"
    ]
].head(15)

,company_id,debt_to_equity,debt_risk_score
0,ABB,0.022438,0
1,ADANIENSOL,2.932521,3
2,ADANIENT,1.671358,2
3,ADANIGREEN,6.595282,3
4,ADANIPORTS,0.937322,1
5,ADANIPOWER,0.802318,1
6,AMBUJACEM,0.016861,0
7,APOLLOHOSP,0.768887,1
8,ASIANPAINT,0.132102,0
9,AXISBANK,8.249123,3


In [7]:
if "return_on_equity_pct" in latest_data.columns:
    latest_data["profitability_risk_score"] = np.select(
        [
            latest_data["return_on_equity_pct"].isna(),
            latest_data["return_on_equity_pct"] < 0,
            latest_data["return_on_equity_pct"] < 5,
            latest_data["return_on_equity_pct"] < 15,
            latest_data["return_on_equity_pct"] >= 15
        ],
        [
            2,
            3,
            2,
            1,
            0
        ],
        default=2
    )

latest_data[
    [
        "company_id",
        "return_on_equity_pct",
        "profitability_risk_score"
    ]
].head(15)

,company_id,return_on_equity_pct,profitability_risk_score
0,ABB,32.468235,0
1,ADANIENSOL,9.461277,1
2,ADANIENT,8.534650,1
3,ADANIGREEN,12.812691,1
4,ADANIPORTS,15.354883,0
5,ADANIPOWER,48.276741,0
6,AMBUJACEM,11.428985,1
7,APOLLOHOSP,13.480392,1
8,ASIANPAINT,29.677488,0
9,AXISBANK,15.832712,0


In [8]:
if "interest_coverage" in latest_data.columns:
    latest_data["interest_risk_score"] = np.select(
        [
            latest_data["interest_coverage"].isna(),
            latest_data["interest_coverage"] <= 1,
            latest_data["interest_coverage"] <= 2,
            latest_data["interest_coverage"] <= 5,
            latest_data["interest_coverage"] > 5
        ],
        [
            2,
            3,
            2,
            1,
            0
        ],
        default=2
    )

latest_data[
    [
        "company_id",
        "interest_coverage",
        "interest_risk_score"
    ]
].head(15)

,company_id,interest_coverage,interest_risk_score
0,ABB,121.083333,0
1,ADANIENSOL,2.063968,1
2,ADANIENT,2.497695,1
3,ADANIGREEN,1.461846,2
4,ADANIPORTS,5.703988,0
5,ADANIPOWER,5.380165,0
6,AMBUJACEM,23.188406,0
7,APOLLOHOSP,5.331849,0
8,ASIANPAINT,37.000000,0
9,AXISBANK,1.690714,2


In [9]:
if "asset_turnover" in latest_data.columns:
    latest_data["efficiency_risk_score"] = np.select(
        [
            latest_data["asset_turnover"].isna(),
            latest_data["asset_turnover"] < 0.5,
            latest_data["asset_turnover"] < 1.0,
            latest_data["asset_turnover"] < 2.0,
            latest_data["asset_turnover"] >= 2.0
        ],
        [
            2,
            3,
            2,
            1,
            0
        ],
        default=2
    )

latest_data[
    [
        "company_id",
        "asset_turnover",
        "efficiency_risk_score"
    ]
].head(15)

,company_id,asset_turnover,efficiency_risk_score
0,ABB,1.126324,1
1,ADANIENSOL,0.283696,3
2,ADANIENT,0.600432,2
3,ADANIGREEN,0.104670,3
4,ADANIPORTS,0.228301,3
5,ADANIPOWER,0.547240,2
6,AMBUJACEM,0.508114,2
7,APOLLOHOSP,1.138394,1
8,ASIANPAINT,1.187084,1
9,AXISBANK,0.072037,3


In [10]:
if "financial_momentum_score" in latest_data.columns:
    latest_data["momentum_risk_score"] = np.select(
        [
            latest_data["financial_momentum_score"].isna(),
            latest_data["financial_momentum_score"] <= -2,
            latest_data["financial_momentum_score"] < 0,
            latest_data["financial_momentum_score"] <= 2,
            latest_data["financial_momentum_score"] > 2
        ],
        [
            2,
            3,
            2,
            1,
            0
        ],
        default=2
    )

latest_data[
    [
        "company_id",
        "financial_momentum_score",
        "momentum_risk_score"
    ]
].head(15)

,company_id,financial_momentum_score,momentum_risk_score
0,ABB,2,1
1,ADANIENSOL,0,1
2,ADANIENT,-1,2
3,ADANIGREEN,1,1
4,ADANIPORTS,3,0
5,ADANIPOWER,3,0
6,AMBUJACEM,-3,3
7,APOLLOHOSP,0,1
8,ASIANPAINT,-1,2
9,AXISBANK,2,1


In [11]:
risk_score_columns = [
    col
    for col in [
        "debt_risk_score",
        "profitability_risk_score",
        "interest_risk_score",
        "efficiency_risk_score",
        "momentum_risk_score"
    ]
    if col in latest_data.columns
]

print("Risk Score Columns:")
print(risk_score_columns)

Risk Score Columns:
['debt_risk_score', 'profitability_risk_score', 'interest_risk_score', 'efficiency_risk_score', 'momentum_risk_score']


In [12]:
latest_data["total_risk_score"] = (
    latest_data[risk_score_columns]
    .sum(axis=1)
)

latest_data[
    [
        "company_id"
    ]
    + risk_score_columns
    + ["total_risk_score"]
].head(20)

,company_id,debt_risk_score,profitability_risk_score,interest_risk_score,efficiency_risk_score,momentum_risk_score,total_risk_score
0,ABB,0,0,0,1,1,2
1,ADANIENSOL,3,1,1,3,1,9
2,ADANIENT,2,1,1,2,2,8
3,ADANIGREEN,3,1,2,3,1,10
4,ADANIPORTS,1,0,0,3,0,4
5,ADANIPOWER,1,0,0,2,0,3
6,AMBUJACEM,0,1,0,2,3,6
7,APOLLOHOSP,1,1,0,1,1,4
8,ASIANPAINT,0,0,0,1,2,3
9,AXISBANK,3,0,2,3,1,9


In [13]:
max_possible_risk = len(risk_score_columns) * 3

latest_data["normalized_risk_score"] = (
    latest_data["total_risk_score"]
    / max_possible_risk
) * 100

latest_data["normalized_risk_score"] = (
    latest_data["normalized_risk_score"]
    .round(2)
)

latest_data[
    [
        "company_id",
        "total_risk_score",
        "normalized_risk_score"
    ]
].head(20)

,company_id,total_risk_score,normalized_risk_score
0,ABB,2,13.33
1,ADANIENSOL,9,60.00
2,ADANIENT,8,53.33
3,ADANIGREEN,10,66.67
4,ADANIPORTS,4,26.67
5,ADANIPOWER,3,20.00
6,AMBUJACEM,6,40.00
7,APOLLOHOSP,4,26.67
8,ASIANPAINT,3,20.00
9,AXISBANK,9,60.00


In [14]:
latest_data["risk_category"] = pd.cut(
    latest_data["normalized_risk_score"],
    bins=[
        -np.inf,
        25,
        50,
        75,
        np.inf
    ],
    labels=[
        "Low Risk",
        "Moderate Risk",
        "High Risk",
        "Critical Risk"
    ],
    include_lowest=True
)

latest_data[
    [
        "company_id",
        "normalized_risk_score",
        "risk_category"
    ]
].head(20)

,company_id,normalized_risk_score,risk_category
0,ABB,13.33,Low Risk
1,ADANIENSOL,60.00,High Risk
2,ADANIENT,53.33,High Risk
3,ADANIGREEN,66.67,High Risk
4,ADANIPORTS,26.67,Moderate Risk
5,ADANIPOWER,20.00,Low Risk
6,AMBUJACEM,40.00,Moderate Risk
7,APOLLOHOSP,26.67,Moderate Risk
8,ASIANPAINT,20.00,Low Risk
9,AXISBANK,60.00,High Risk


In [15]:
latest_data["risk_rank"] = (
    latest_data["normalized_risk_score"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)

latest_data[
    [
        "company_id",
        "normalized_risk_score",
        "risk_category",
        "risk_rank"
    ]
].sort_values("risk_rank").head(20)

,company_id,normalized_risk_score,risk_category,risk_rank
56,KOTAKBANK,73.33,High Risk,1
77,SHRIRAMFIN,73.33,High Risk,1
91,UNIONBANK,73.33,High Risk,1
3,ADANIGREEN,66.67,High Risk,2
37,HDFCBANK,66.67,High Risk,2
46,INDUSINDBK,66.67,High Risk,2
54,JSWENERGY,66.67,High Risk,2
17,BHEL,66.67,High Risk,2
69,PFC,66.67,High Risk,2
83,TATASTEEL,66.67,High Risk,2


In [16]:
company_risk_report = latest_data[
    [
        "company_id",
        "financial_year",
        "return_on_equity_pct",
        "debt_to_equity",
        "interest_coverage",
        "asset_turnover",
        "financial_momentum_score"
    ]
    + risk_score_columns
    + [
        "total_risk_score",
        "normalized_risk_score",
        "risk_category",
        "risk_rank"
    ]
].copy()

company_risk_report = company_risk_report.sort_values(
    "risk_rank"
)

company_risk_report.head(20)

,company_id,financial_year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,financial_momentum_score,debt_risk_score,profitability_risk_score,interest_risk_score,efficiency_risk_score,momentum_risk_score,total_risk_score,normalized_risk_score,risk_category,risk_rank
56,KOTAKBANK,2024,14.013018,4.003739,1.237006,0.073257,-1,3,1,2,3,2,11,73.33,High Risk,1
77,SHRIRAMFIN,2024,15.116350,3.994034,0.675071,0.146569,-1,3,0,3,3,2,11,73.33,High Risk,1
91,UNIONBANK,2024,14.136560,12.823705,0.072439,0.071595,2,3,1,3,3,1,11,73.33,High Risk,1
3,ADANIGREEN,2024,12.812691,6.595282,1.461846,0.104670,1,3,1,2,3,1,10,66.67,High Risk,2
37,HDFCBANK,2024,14.339740,6.808787,1.400897,0.070381,1,3,1,2,3,1,10,66.67,High Risk,2
46,INDUSINDBK,2024,14.252273,6.885743,1.878781,0.088842,1,3,1,2,3,1,10,66.67,High Risk,2
54,JSWENERGY,2024,8.280530,1.515601,2.621529,0.198833,-2,2,1,1,3,3,10,66.67,High Risk,2
17,BHEL,2024,1.153941,0.362386,0.858696,0.399629,-1,0,2,3,3,2,10,66.67,High Risk,2
69,PFC,2024,26.160934,8.521864,0.579224,0.088084,1,3,0,3,3,1,10,66.67,High Risk,2
83,TATASTEEL,2024,-5.334927,0.946184,2.963239,0.850950,-3,1,3,1,2,3,10,66.67,High Risk,2


In [17]:
high_risk_companies = company_risk_report[
    company_risk_report["risk_category"].isin(
        [
            "High Risk",
            "Critical Risk"
        ]
    )
].copy()

high_risk_companies = high_risk_companies.sort_values(
    "normalized_risk_score",
    ascending=False
)

print("High/Critical Risk Companies:", len(high_risk_companies))

high_risk_companies.head(20)

High/Critical Risk Companies: 24


,company_id,financial_year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,financial_momentum_score,debt_risk_score,profitability_risk_score,interest_risk_score,efficiency_risk_score,momentum_risk_score,total_risk_score,normalized_risk_score,risk_category,risk_rank
56,KOTAKBANK,2024,14.013018,4.003739,1.237006,0.073257,-1,3,1,2,3,2,11,73.33,High Risk,1
77,SHRIRAMFIN,2024,15.116350,3.994034,0.675071,0.146569,-1,3,0,3,3,2,11,73.33,High Risk,1
91,UNIONBANK,2024,14.136560,12.823705,0.072439,0.071595,2,3,1,3,3,1,11,73.33,High Risk,1
3,ADANIGREEN,2024,12.812691,6.595282,1.461846,0.104670,1,3,1,2,3,1,10,66.67,High Risk,2
37,HDFCBANK,2024,14.339740,6.808787,1.400897,0.070381,1,3,1,2,3,1,10,66.67,High Risk,2
46,INDUSINDBK,2024,14.252273,6.885743,1.878781,0.088842,1,3,1,2,3,1,10,66.67,High Risk,2
54,JSWENERGY,2024,8.280530,1.515601,2.621529,0.198833,-2,2,1,1,3,3,10,66.67,High Risk,2
17,BHEL,2024,1.153941,0.362386,0.858696,0.399629,-1,0,2,3,3,2,10,66.67,High Risk,2
69,PFC,2024,26.160934,8.521864,0.579224,0.088084,1,3,0,3,3,1,10,66.67,High Risk,2
83,TATASTEEL,2024,-5.334927,0.946184,2.963239,0.850950,-3,1,3,1,2,3,10,66.67,High Risk,2


In [18]:
low_risk_companies = company_risk_report[
    company_risk_report["risk_category"] == "Low Risk"
].copy()

low_risk_companies = low_risk_companies.sort_values(
    "normalized_risk_score"
)

print("Low Risk Companies:", len(low_risk_companies))

low_risk_companies.head(20)

Low Risk Companies: 28


,company_id,financial_year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,financial_momentum_score,debt_risk_score,profitability_risk_score,interest_risk_score,efficiency_risk_score,momentum_risk_score,total_risk_score,normalized_risk_score,risk_category,risk_rank
45,INDIGO,2024,892.568306,0.018579,22.602322,56.386252,1,0,0,0,0,1,1,6.67,Low Risk,11
15,BEL,2024,4744.047619,0.511905,420.916667,125.888199,3,1,0,0,0,0,1,6.67,Low Risk,11
65,NESTLEIND,2024,117.754491,0.103293,40.089655,2.318160,1,0,0,0,0,1,1,6.67,Low Risk,11
88,TRENT,2024,36.307768,0.430924,5.355978,1.727869,4,0,0,0,1,0,1,6.67,Low Risk,11
0,ABB,2024,32.468235,0.022438,121.083333,1.126324,2,0,0,0,1,1,2,13.33,Low Risk,10
19,BPCL,2024,35.511337,0.721875,10.624729,2.213652,2,1,0,0,0,1,2,13.33,Low Risk,10
70,PIDILITIND,2024,20.780302,0.045438,53.098039,1.025422,2,0,0,0,1,1,2,13.33,Low Risk,10
84,TCS,2024,50.944314,0.088641,82.642674,1.655941,2,0,0,0,1,1,2,13.33,Low Risk,10
62,MARUTI,2024,15.750385,0.001390,96.010309,1.230296,2,0,0,0,1,1,2,13.33,Low Risk,10
39,HEROMOTOCO,2024,21.142437,0.034239,47.248186,1.444920,0,0,0,0,1,1,2,13.33,Low Risk,10


In [19]:
risk_distribution = (
    company_risk_report["risk_category"]
    .value_counts(dropna=False)
    .rename_axis("risk_category")
    .reset_index(name="company_count")
)

risk_distribution

,risk_category,company_count
0,Moderate Risk,40
1,Low Risk,28
2,High Risk,24
3,Critical Risk,0


In [20]:
risk_driver_names = {
    "debt_risk_score": "Debt Risk",
    "profitability_risk_score": "Profitability Risk",
    "interest_risk_score": "Interest Coverage Risk",
    "efficiency_risk_score": "Asset Efficiency Risk",
    "momentum_risk_score": "Financial Momentum Risk"
}

latest_data["primary_risk_driver"] = (
    latest_data[risk_score_columns]
    .idxmax(axis=1)
    .map(risk_driver_names)
)

latest_data[
    [
        "company_id",
        "normalized_risk_score",
        "risk_category",
        "primary_risk_driver"
    ]
].head(20)

,company_id,normalized_risk_score,risk_category,primary_risk_driver
0,ABB,13.33,Low Risk,Asset Efficiency Risk
1,ADANIENSOL,60.00,High Risk,Debt Risk
2,ADANIENT,53.33,High Risk,Debt Risk
3,ADANIGREEN,66.67,High Risk,Debt Risk
4,ADANIPORTS,26.67,Moderate Risk,Asset Efficiency Risk
5,ADANIPOWER,20.00,Low Risk,Asset Efficiency Risk
6,AMBUJACEM,40.00,Moderate Risk,Financial Momentum Risk
7,APOLLOHOSP,26.67,Moderate Risk,Debt Risk
8,ASIANPAINT,20.00,Low Risk,Financial Momentum Risk
9,AXISBANK,60.00,High Risk,Debt Risk


In [21]:
company_risk_report = company_risk_report.merge(
    latest_data[
        [
            "company_id",
            "primary_risk_driver"
        ]
    ],
    on="company_id",
    how="left"
)

company_risk_report[
    [
        "company_id",
        "normalized_risk_score",
        "risk_category",
        "primary_risk_driver",
        "risk_rank"
    ]
].head(20)

,company_id,normalized_risk_score,risk_category,primary_risk_driver,risk_rank
0,KOTAKBANK,73.33,High Risk,Debt Risk,1
1,SHRIRAMFIN,73.33,High Risk,Debt Risk,1
2,UNIONBANK,73.33,High Risk,Debt Risk,1
3,ADANIGREEN,66.67,High Risk,Debt Risk,2
4,HDFCBANK,66.67,High Risk,Debt Risk,2
5,INDUSINDBK,66.67,High Risk,Debt Risk,2
6,JSWENERGY,66.67,High Risk,Asset Efficiency Risk,2
7,BHEL,66.67,High Risk,Interest Coverage Risk,2
8,PFC,66.67,High Risk,Debt Risk,2
9,TATASTEEL,66.67,High Risk,Profitability Risk,2


In [22]:
risk_driver_distribution = (
    company_risk_report["primary_risk_driver"]
    .value_counts(dropna=False)
    .rename_axis("primary_risk_driver")
    .reset_index(name="company_count")
)

risk_driver_distribution

,primary_risk_driver,company_count
0,Asset Efficiency Risk,42
1,Debt Risk,31
2,Financial Momentum Risk,13
3,Profitability Risk,4
4,Interest Coverage Risk,2


In [23]:
os.makedirs(REPORT_DIR, exist_ok=True)

company_risk_report.to_csv(
    "../reports/day20_company_risk_report.csv",
    index=False
)

high_risk_companies.to_csv(
    "../reports/day20_high_risk_companies.csv",
    index=False
)

low_risk_companies.to_csv(
    "../reports/day20_low_risk_companies.csv",
    index=False
)

risk_distribution.to_csv(
    "../reports/day20_risk_distribution.csv",
    index=False
)

risk_driver_distribution.to_csv(
    "../reports/day20_risk_driver_distribution.csv",
    index=False
)

print("Day 20 risk reports saved successfully.")

Day 20 risk reports saved successfully.


In [24]:
print("=" * 65)
print("DAY 20 - FINANCIAL RISK ANALYSIS")
print("=" * 65)

print("\nCompanies Analyzed:")
print(company_risk_report["company_id"].nunique())

print("\nRisk Metrics:")
print(risk_score_columns)

print("\nRISK DISTRIBUTION")
print(risk_distribution)

print("\nPRIMARY RISK DRIVERS")
print(risk_driver_distribution)

print("\nTOP 10 HIGHEST RISK COMPANIES")
print(
    company_risk_report[
        [
            "company_id",
            "normalized_risk_score",
            "risk_category",
            "primary_risk_driver",
            "risk_rank"
        ]
    ].head(10)
)

print("\n" + "=" * 65)
print("DAY 20 COMPLETED SUCCESSFULLY")
print("=" * 65)

DAY 20 - FINANCIAL RISK ANALYSIS

Companies Analyzed:
92

Risk Metrics:
['debt_risk_score', 'profitability_risk_score', 'interest_risk_score', 'efficiency_risk_score', 'momentum_risk_score']

RISK DISTRIBUTION
   risk_category  company_count
0  Moderate Risk             40
1       Low Risk             28
2      High Risk             24
3  Critical Risk              0

PRIMARY RISK DRIVERS
       primary_risk_driver  company_count
0    Asset Efficiency Risk             42
1                Debt Risk             31
2  Financial Momentum Risk             13
3       Profitability Risk              4
4   Interest Coverage Risk              2

TOP 10 HIGHEST RISK COMPANIES
   company_id  normalized_risk_score risk_category     primary_risk_driver  risk_rank
0   KOTAKBANK                  73.33     High Risk               Debt Risk          1
1  SHRIRAMFIN                  73.33     High Risk               Debt Risk          1
2   UNIONBANK                  73.33     High Risk               De